# 12. 3x3 median filter, omega 1.5 harmonics, seasonality filter and WorldCover mask

## Setup

In [ ]:
from pathlib import Path
import json
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").is_file():
    if root == root.parent:
        raise RuntimeError("Open this notebook inside the project repository")
    root = root.parent
if str(root / "src") not in sys.path:
    sys.path.insert(0, str(root / "src"))
if str(root) not in sys.path:
    sys.path.insert(0, str(root))  # experiments package

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from experiments.pilots import climate, clustering, data, earth_engine, evaluation, features, full
from experiments.tuning import tuning
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
AREA = "pilots"


def show_and_save(figure, run, name):
    figure.tight_layout()
    figure.savefig(run / f"{name}.png", dpi=150)
    display(figure)
    plt.close(figure)


def repeated(name, X, rows, ok, seeds=(1, 2, 3), params=evaluation.PARAMS):
    """Mean and SD over random-forest seeds of every evaluation view."""
    summaries, details = [], []
    for seed in seeds:
        summary, holdout = evaluation.evaluate(name, X, rows, ok, params, seed)
        summaries.append({**summary, "seed": seed})
        details.extend({**r, "seed": seed} for r in holdout)
    return summaries, details


def summarize(summaries):
    table = pd.DataFrame(summaries)
    metrics = ["random_split_macro_f1", "spatial_cv_macro_f1", "dataset_holdout_macro_f1", "jecam_macro_f1",
               "jecam_carrot_f1", "jecam_cabbage_f1", "jecam_potato_f1"]
    grouped = table.groupby("feature_set", sort=False)
    out = grouped[metrics].agg(["mean", "std"]).round(3)
    out.insert(0, ("pixels_used", ""), grouped["pixels_used"].first())
    return out

## Inputs

In [ ]:
full_run = ""  # Optional pipeline_outputs/pilots/subset_<id>; blank uses the latest
full_run = Path(full_run) if full_run else tuning.latest_run(root, "full_data", area=AREA)
rows, arrays = data.load(full_run)
cube = features.stack(arrays["s2_values"])
valid = arrays["s2_valid"]
print("Full data:", full_run, "| pixels:", len(rows))

In [ ]:
settings = json.loads((root / "settings.local.json").read_text())
inputs = json.loads((full_run / "inputs.json").read_text())
quality_handoff, quality = data.load_quality(root, inputs["development_features"])
regions = data.sample_parcels(quality["regions"], {"train": None, "validation": None, "test": None})
pixels_per_parcel = int(inputs["pixels_per_parcel"])
print("Quality handoff:", quality_handoff, "| pixels per parcel:", pixels_per_parcel)

## 3x3 median series inside each parcel fragment

In [ ]:
run = tuning.new_run(root, "filters", full_run / "meta.json", area=AREA)
median_rows, median_values, median_valid = data.load_series(regions, quality["pixel_paths"], pixels_per_parcel, neighbourhood=1)
key = lambda r: (r["region_id"], r["source_crs"], r["x"], r["y"])
if [key(r) for r in median_rows] != [key(r) for r in rows]:
    raise ValueError("Filtered pixels do not line up with the full-data pixels")
s2, s1 = arrays["s2_values"], arrays["s1_values"]
print("Mean clear half-months: raw", round(float(valid.sum(1).mean()), 1), "| 3x3 median", round(float(median_valid.sum(1).mean()), 1))

## Feature sets

In [ ]:
all_radar = full.combine(full.s1_raw, full.s1_ratios)
sets = {
    "s2_harmonic": full.s2_harmonic(s2, valid),
    "s2_harmonic_3x3": full.s2_harmonic(median_values, median_valid),
    "s2_harmonic_omega1.5": full.s2_harmonic_omega(1.5)(s2, valid),
    "s2_harmonic_3x3_omega1.5": full.s2_harmonic_omega(1.5)(median_values, median_valid),
    "s2_harmonic_plus_all_radar": full.combine(full.s2_harmonic, full.s1_raw, full.s1_ratios)(s2, valid, s1),
    "s2_harmonic_3x3_plus_all_radar": full.combine(full.s2_harmonic, full.s1_raw, full.s1_ratios)(median_values, median_valid, s1),
}
common = np.logical_and.reduce([ok for _, ok, _ in sets.values()])
display(pd.DataFrame([{"feature_set": k, "predictors": X.shape[1], "pixels": int(ok.sum())} for k, (X, ok, _) in sets.items()]))
print("Pixels usable by every set:", int(common.sum()))

## Evaluate

In [ ]:
seeds = (1, 2)
summaries, details = [], []
for name, (X, _, _) in sets.items():
    s, d = repeated(name, np.nan_to_num(X), rows, common, seeds)
    summaries += s
    details += d
    print(name, "done", flush=True)
tuning.save_records(run, "summary", summaries)
tuning.save_records(run, "dataset_holdout", details)
display(summarize(summaries))

## Paoay pixels: raw and 3x3 median

In [ ]:
paoay = json.loads((full_run / "paoay.json").read_text())
grid = paoay["grid"]
mask = arrays["paoay_mask"]
pixels, p_s2, p_valid, p_s1 = full.paoay_pixels(arrays)
median_cube = features.grid_median(arrays["paoay_cube"].astype(float) * .0001)
p_median = median_cube[:, :, mask].transpose(2, 0, 1)
p_median_valid = np.isfinite(p_median).all(axis=2)
print("Paoay pixels:", len(pixels), "| mean clear half-months raw", round(float(p_valid.sum(1).mean()), 1),
      "| 3x3", round(float(p_median_valid.sum(1).mean()), 1))

## Three cropland masks

In [ ]:
import rasterio
from rasterio.warp import transform as reproject_points
from crop_pipeline.analysis_workflow import resolve_stage_pointer
handoff = resolve_stage_pointer(root / "sen4agrinet_work/paoay/land_activity/land_activity_stage.json")
centres = [rasterio.Affine(*grid["transform"]) * (c + .5, r + .5) for r, c in pixels]
with rasterio.open(handoff.parent / "land_activity.tif") as raster:
    xs, ys = reproject_points(grid["crs"], raster.crs, [x for x, _ in centres], [y for _, y in centres])
    land_activity_cropland = np.asarray([v[0] for v in raster.sample(zip(xs, ys))]) == 1
ee = earth_engine.initialize(settings["earth_engine_project"])
worldcover = earth_engine.worldcover_grid(ee, grid, run / "cache")[mask]
worldcover_cropland = worldcover == earth_engine.WORLDCOVER_CROPLAND
crop_ratio, crop_ok = features.seasonality_ratio(s2, valid)
development = np.asarray([r["partition"] in ("train", "validation") for r in rows])
threshold = float(np.nanpercentile(crop_ratio[development & crop_ok], 5))
p_ratio, p_ratio_ok = features.seasonality_ratio(p_s2, p_valid)
seasonal = p_ratio_ok & (p_ratio >= threshold)
masks = {"land_activity": land_activity_cropland, "worldcover": worldcover_cropland, "seasonality_ratio": seasonal}
print("Seasonality threshold (5th percentile of training crop pixels):", round(threshold, 3))
display(pd.DataFrame([{"mask": k, "cropland_share": round(float(m.mean()), 3)} for k, m in masks.items()]))
wc_names = {10: "tree cover", 20: "shrubland", 30: "grassland", 40: "cropland", 50: "built-up", 60: "bare",
            80: "water", 90: "wetland", 95: "mangroves", 100: "moss"}
display(pd.Series([wc_names.get(int(v), str(int(v))) for v in worldcover]).value_counts(normalize=True).round(3).rename("worldcover_share").to_frame())
agreement = pd.DataFrame({a: {b: round(float((masks[a] == masks[b]).mean()), 3) for b in masks} for a in masks})
display(agreement)

## Apply each feature set to Paoay under each mask

In [ ]:
rule = {"minimum_top_probability": .6, "minimum_margin": .1}
builders = {
    "s2_harmonic": (full.s2_harmonic, False),
    "s2_harmonic_3x3": (full.s2_harmonic, True),
    "s2_harmonic_omega1.5": (full.s2_harmonic_omega(1.5), False),
    "s2_harmonic_3x3_omega1.5": (full.s2_harmonic_omega(1.5), True),
    "s2_harmonic_plus_all_radar": (full.combine(full.s2_harmonic, full.s1_raw, full.s1_ratios), False),
    "s2_harmonic_3x3_plus_all_radar": (full.combine(full.s2_harmonic, full.s1_raw, full.s1_ratios), True),
}
statistics = pd.read_csv(root / "data/paoay/benguet_vegetable_area_2024.csv")
benguet = dict(zip(statistics.crop, statistics.area_ha / statistics.area_ha.sum()))
records, labels = [], {}
for name, (build, filtered) in builders.items():
    X, ok, _ = build(median_values if filtered else s2, median_valid if filtered else valid, s1)
    pX, pok, _ = build(p_median if filtered else p_s2, p_median_valid if filtered else p_valid, p_s1)
    index = np.nonzero(development & ok)[0]
    model = evaluation.fit_rows(np.nan_to_num(X), rows, index, {**evaluation.PARAMS, "n_estimators": 500})
    probabilities = model.predict_proba(np.nan_to_num(pX))
    ordered = np.sort(probabilities, axis=1)
    accepted = pok & (ordered[:, -1] >= rule["minimum_top_probability"]) & (ordered[:, -1] - ordered[:, -2] >= rule["minimum_margin"])
    label = np.where(~pok, "No features", np.where(accepted, model.classes_[np.argmax(probabilities, axis=1)], "Unknown crop type"))
    labels[name] = label
    for mask_name, cropland in masks.items():
        confident = pd.Series(label[cropland & accepted]).value_counts(normalize=True)
        records.append({"feature_set": name, "mask": mask_name, "cropland_pixels": int(cropland.sum()),
                        "confident_share": float(accepted[cropland].mean()),
                        **{f"{c}_share": float(confident.get(c, 0)) for c in evaluation.CROPS}})
    print(name, "applied", flush=True)
paoay_table = pd.DataFrame(records)
tuning.save_records(run, "paoay_shares", records)
display(paoay_table.round(3))
print("Benguet 2024 shares:", {k: round(v, 3) for k, v in benguet.items()})

## Reference areas

In [ ]:
from crop_pipeline.paoay import aligned
from rasterio.features import geometry_mask
from shapely.geometry import mapping
_, areas, _ = aligned.study_area([Path(p) for p in paoay["kmz_paths"]], settings.get("paoay_roi_name", "Region of Interest"),
                                 list(settings.get("paoay_context_names", [])), grid["crs"])
area_rows = []
for name, label in labels.items():
    raster = np.full(mask.shape, "", dtype=object)
    raster[tuple(pixels.T)] = label
    for area in areas:
        inside = ~geometry_mask([mapping(area["geometry"])], out_shape=mask.shape, transform=rasterio.Affine(*grid["transform"])) & mask
        area_rows.append({"feature_set": name, "area": area["name"], **pd.Series(raster[inside]).value_counts().to_dict()})
area_table = pd.DataFrame(area_rows).fillna(0)
area_table.to_csv(run / "reference_areas.csv", index=False)
display(area_table)

## Masks and maps

In [ ]:
figure, axes = plt.subplots(1, 3, figsize=(15, 5))
for axis, (name, cropland) in zip(axes, masks.items()):
    image = np.full(mask.shape, np.nan)
    image[tuple(pixels.T)] = cropland.astype(float)
    axis.imshow(image, cmap="Greens", vmin=0, vmax=1.3, interpolation="nearest")
    axis.set_title(f"{name}: {cropland.mean():.0%} cropland", fontsize=10)
    axis.axis("off")
show_and_save(figure, run, "cropland_masks")
codes = {"carrot": 1, "cabbage": 2, "potato": 3, "Unknown crop type": 4, "No features": 5}
figure, axes = plt.subplots(2, 3, figsize=(15, 10))
for axis, (name, label) in zip(axes.ravel(), labels.items()):
    image = np.full(mask.shape, np.nan)
    image[tuple(pixels.T)] = [codes[l] for l in label]
    axis.imshow(image, cmap="Set2", vmin=1, vmax=8, interpolation="nearest")
    axis.set_title(name, fontsize=9)
    axis.axis("off")
show_and_save(figure, run, "paoay_maps")
print("Run folder:", run)